# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 

In [2]:
import os
price_data_path = os.getenv("PRICE_DATA")
price_data_path #the environment variable is loaded into a variable

'../../05_src/data/prices/'

In [3]:
import dask.dataframe as dd

c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
import os
from glob import glob

# Write your code below.
parquet_files = glob(os.path.join(price_data_path, '**/*.parquet'), recursive=True)

dd_px = dd.read_parquet(parquet_files).set_index("Ticker")

dd_px.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
A,2000-01-03 00:00:00+00:00,43.382847,51.502148,56.464592,48.193848,56.330471,4674353.0,2000
A,2000-01-04 00:00:00+00:00,40.068882,47.567955,49.266811,46.316166,48.730328,4765083.0,2000
A,2000-01-05 00:00:00+00:00,37.583393,44.617310,47.567955,43.141991,47.389126,5758642.0,2000
A,2000-01-06 00:00:00+00:00,36.152378,42.918453,44.349072,41.577251,44.080830,2534434.0,2000
A,2000-01-07 00:00:00+00:00,39.165066,46.494991,47.165592,42.203148,42.247852,2819626.0,2000


In [ ]:
dd_px.index.unique().compute() #Here I wanted to confirm that there are multiple tickers so I can then group them

c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(


c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pyarrow\pandas_compat.py:766: DeprecationWarning: DatetimeTZBlock is deprecated and will be removed in a future version. Use public APIs instead.
  klass=_int.DatetimeTZBlock,
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\

Index(['BEN', 'KDP', 'NTAP', 'TT', 'DFS', 'KR', 'ETSY', 'EVRG', 'STLD', 'ALLE',
       ...
       'TXT', 'DOV', 'ED', 'FOX', 'IP', 'IQV', 'FRT', 'JPM', 'PLD', 'VTR'],
      dtype='object', name='Ticker', length=503)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [7]:
# Write your code below.
#Adding the lags for the two columns
dd_shift = dd_px.groupby("Ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x['Close'].shift(1),         # Lag for Close
        Adj_Close_lag_1=x['Adj Close'].shift(1)  # Lag for Adj_Close
    )
)

C:\Users\carva\AppData\Local\Temp\ipykernel_19104\1411561393.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby("Ticker", group_keys=False).apply(


In [8]:
#Adding the return columns
dd_rets = dd_shift.assign(
    Returns=lambda x: x['Close'] / x['Close_lag_1'] - 1  # Returns calculation
)

In [9]:
#Adding the hi_lo_range col and assigning it to a new df
dd_feat = dd_rets.assign(
    hi_lo_range=lambda x: x['High'] - x['Low']  # High minus Low
)

In [15]:
dd_feat.sample(frac=0.0001).compute()  #Here I want to visualize a sample of the new data frame. I see that there are some NAN rows

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
Ticker,,,,,,,,,,,,
DOV,2018-04-20 00:00:00+00:00,69.561241,77.560585,77.915993,76.882065,77.560585,1631065.0,2018,77.544426,69.546761,0.000208,1.033928
CBRE,2001-11-20 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2001,NaN,NaN,NaN,NaN
FANG,2006-06-27 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2006,NaN,NaN,NaN,NaN
DTE,2015-09-11 00:00:00+00:00,47.116814,64.629784,64.638298,63.455318,63.693619,877960.0,2015,63.889362,46.577023,0.011589,1.182980
AOS,2023-01-19 00:00:00+00:00,55.843193,57.790001,59.880001,57.720001,59.880001,1045300.0,2023,60.139999,58.114029,-0.039075,2.160000
...,...,...,...,...,...,...,...,...,...,...,...,...
SBAC,2009-05-18 00:00:00+00:00,22.634996,24.010000,24.020000,22.840000,23.000000,1876500.0,2009,22.200001,20.928652,0.081532,1.180000
VLTO,2014-02-07 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2014,NaN,NaN,NaN,NaN
TFC,2018-12-19 00:00:00+00:00,33.268089,43.299999,44.500000,42.959999,44.009998,6380300.0,2018,44.160000,33.928844,-0.019475,1.540001


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [25]:
# Write your code below.



Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.